In [0]:
import pyspark.pandas as ps
import pandas as pd
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.feature_selection import SelectKBest, f_classif, f_regression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix


In [0]:
# load tables
activiteiten = spark.table("default.cleaned_activiteiten")
ozp = spark.table("default.cleaned_ozp")
verblijf = spark.table("default.cleaned_verblijf")
productiviteit = spark.table("default.cleaned_productiviteit")
diensten = spark.table("default.cleaned_diensten")
tarieven = spark.table("default.cleaned_tarieven")

In [0]:
# aansluiting met VC - ZPM omzet 2025 consult = 80.038.091, groep = 4.654.825, OZP = 3.568.317, Toeslag consult = 3.495.944, Verblijf = 41.725.616
f1_activiteiten = activiteiten.filter(F.col("jaar") == 2025) \
                     .filter(F.col("declarabel") == True) \
                     .filter(F.col("soort_ggz_zorg") == "ZVW") \
                     .groupBy("declaratiecode_categorie") \
                     .agg(F.sum("bedrag_totaal"))

f1_ozp = ozp.filter(F.col("jaar") == 2025) \
                  .filter(F.col("declarabel") == True) \
                  .filter(F.col("soort_ggz_zorg") == "ZVW") \
                  .filter(F.col("crisis_binnen_budget") == False) \
                  .groupBy("declaratiecode_categorie") \
                  .agg(F.sum("bedrag_totaal"))

f1_verblijf = verblijf.filter(F.col("jaar") == 2025) \
                            .filter(F.col("declarabel") == True) \
                            .filter(F.col("soort_ggz_zorg") == "ZVW") \
                            .groupBy("declaratiecode_categorie") \
                            .agg(F.sum("bedrag_totaal"))

f1_activiteiten.show()
f1_ozp.show()
f1_verblijf.show()

In [0]:
# aansluiting met VC - ZPM Declarabele uren 2025, consult = 249874, groep = 18635, OZP = 2165, Toeslag = 23556
f2_activiteiten = activiteiten.filter(F.col("jaar") == 2025) \
                     .filter(F.col("declarabel") == True) \
                     .filter(F.col("soort_ggz_zorg") == "ZVW") \
                     .groupBy("declaratiecode_categorie") \
                     .agg(F.sum("tijd_client_minuten")/60)

f2_ozp = ozp.filter(F.col("jaar") == 2025) \
                  .filter(F.col("declarabel") == True) \
                  .filter(F.col("soort_ggz_zorg") == "ZVW") \
                  .filter(F.col("crisis_binnen_budget") == False) \
                  .groupBy("declaratiecode_categorie") \
                  .agg(F.sum("tijd_client_minuten")/60)

f2_activiteiten.show()
f2_ozp.show()

In [0]:
# aansluiting met VC - ZPM Unieke Cliënten 17.009
f3_activiteiten = activiteiten.filter(F.col("jaar") == 2025) \
                     .filter(F.col("declarabel") == True) \
                     .filter(F.col("soort_ggz_zorg") == "ZVW") \
                     .select("client_code").distinct()

f3_ozp = ozp.filter(F.col("jaar") == 2025) \
                  .filter(F.col("declarabel") == True) \
                  .filter(F.col("soort_ggz_zorg") == "ZVW") \
                  .filter(F.col("crisis_binnen_budget") == False) \
                  .select("client_code").distinct()

f3_verblijf = verblijf.filter(F.col("jaar") == 2025) \
                    .filter(F.col("declarabel") == True) \
                    .filter(F.col("soort_ggz_zorg") == "ZVW") \
                    .select("client_code").distinct()

f3_combined_clients = f3_activiteiten.union(f3_ozp).union(f3_verblijf).distinct()
total_unique_clients = f3_combined_clients.count()
print(f"Total unique clients in 2025 (ZVW, Declarabel, Non-crisis): {total_unique_clients}")


Now the data has been verified, we can start with defining the target variable: the target variable will be the column tarief_prijspeil_2026. This column contains standardized monetary values. The first step is to aggregate and combine the tarief_prijspeil_2026 per date for the datasets activiteiten, ozp and verblijf because this is the basis for all ZVW revenue. 

In [0]:
# Filtering the datasets for ZVW declarabel no-crisis
activiteiten_zvw = activiteiten.filter(F.col("soort_ggz_zorg") == "ZVW").filter(F.col("declarabel") == True)
ozp_zvw = ozp.filter(F.col("soort_ggz_zorg") == "ZVW").filter(F.col("declarabel") == True).filter(F.col("crisis_binnen_budget") == False)
verblijf_zvw = verblijf.filter(F.col("soort_ggz_zorg") == "ZVW").filter(F.col("declarabel") == True)

# Check if tarief_prijspeil_2026 is the same as bedrag_totaal_nza in 2026 so that we can conclude the tarief_prijspeil_2026 is correct
activiteiten_zvw.filter(F.col("jaar") == 2026).groupBy("jaar").agg(F.sum("bedrag_totaal_nza"), F.sum("tarief_prijspeil_2026")).show()

In [0]:
# Combine the ZVW declarabel no crisis datasets into one
combined_zvw = activiteiten_zvw.select("rapportagedatum", "tarief_prijspeil_2026").union(ozp_zvw.select("rapportagedatum", "tarief_prijspeil_2026")).union(verblijf_zvw.select("rapportagedatum","tarief_prijspeil_2026"))

# Group by date
daily_revenue = (combined_zvw
                 .groupBy("rapportagedatum")
                 .agg(F.round(F.sum("tarief_prijspeil_2026"), 2).alias("totale_omzet"))
                 .orderBy("rapportagedatum"))

# Convert to pandas
df_daily_revenue = daily_revenue.toPandas()
df_daily_revenue['rapportagedatum'] = pd.to_datetime(df_daily_revenue['rapportagedatum'])
df_daily_revenue.info()

In [0]:
# Plot daily revenue
plt.figure(figsize=(15, 6))
sns.lineplot(
    data=df_daily_revenue,
    x='rapportagedatum', 
    y='totale_omzet', 
    color='lightgrey', 
    alpha=0.5,
    label='Daily revenue'
)


# rolling weekly average
df_daily_revenue['omzet_7d_gemiddelde'] = df_daily_revenue['totale_omzet'].rolling(window=7).mean()

sns.lineplot(
    data=df_daily_revenue, 
    x='rapportagedatum', 
    y='omzet_7d_gemiddelde', 
    color='#1f77b4', 
    linewidth=2.5,
    label='7-Day MA'
)

plt.title('ZVW Revenue over time (Tariffs 2026)', fontsize=14, pad=15)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Revenue (€)', fontsize=12)

We can see a strong seasonal trend where holidays have significantly lower revenue, probably because less people will be working on the holidays. Therefore, it would be interesting to investigate the worked hours. We can use the dataset "productiviteit" and the variable "uren_netto".

In [0]:
# Group uren_netto by date
daily_productivity = (productiviteit
                      .groupBy("rapportagedatum")
                      .agg(F.round(F.sum("uren_netto"), 2).alias("totale_uren_netto"))
                      .orderBy("rapportagedatum"))

# Convert to pandas
df_daily_prod = daily_productivity.toPandas()

# Convert to date
df_daily_prod['rapportagedatum'] = pd.to_datetime(df_daily_prod['rapportagedatum'])

# Calculate MA 7-day
df_daily_prod['uren_7d_gemiddelde'] = df_daily_prod['totale_uren_netto'].rolling(window=7).mean()

# Set up plot
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 6))

# Plot daily uren_netto
sns.lineplot(
    data=df_daily_prod, 
    x='rapportagedatum', 
    y='totale_uren_netto', 
    color='lightgreen', 
    alpha=0.5,
    label='Raw worked hours per day'
)

sns.lineplot(
    data=df_daily_prod, 
    x='rapportagedatum', 
    y='uren_7d_gemiddelde', 
    color='#2ca02c',
    linewidth=2.5,
    label='7-Day MA'
)

In 2023 and 2024 we can see that there is a sharp decline of uren_netto during the summer holidays and a slight decline during new years, which indicates that uren_netto could be used to improve the forecast of the ZVW. However, there is also a increasing trend over the years, which should be investigated further. The decline at the end of 2026 is because of expiring contracts from employees; if the contracts have not yet been renewed, it will not be available in the data

To further investigate the increase in uren_netto while revenue stays approximately the same, we will firstly analyze the total direct hours to investigate if the difference in trend is due to a difference in product mix or productivity from employees. Therefore, we can use the tijd_client_minuten from Activiteiten and OZP datasets.

In [0]:
# Select and combine necessary columns
combined_hours = activiteiten_zvw.select("rapportagedatum", "tijd_client_minuten").union(ozp_zvw.select("rapportagedatum", "tijd_client_minuten"))

# Groupby date
direct_hours = (combined_hours
                .groupBy("rapportagedatum")
                .agg(F.round(F.sum("tijd_client_minuten")/60, 2).alias("totale_direct_uren")))

# Convert to pandas
df_direct_hours = direct_hours.toPandas()

# Convert to datetime
df_direct_hours['rapportagedatum'] = pd.to_datetime(df_direct_hours['rapportagedatum'])

# Calculate MA 7-day
df_direct_hours['uren_7d_gemiddelde'] = df_direct_hours['totale_direct_uren'].rolling(window=7).mean()

# Set up plot
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 6))

# Plot daily direct hours
sns.lineplot(
    data=df_direct_hours, 
    x='rapportagedatum', 
    y='totale_direct_uren', 
    color='lightblue', 
    alpha=0.5,
    label='Raw direct hours per day'
)

sns.lineplot(
    data=df_direct_hours, 
    x='rapportagedatum', 
    y='uren_7d_gemiddelde', 
    color='#d62728',
    linewidth=2.5,
    label='7-Day MA'
)


This plot is really similar to the Revenue, which indicates that the productivity is declining. Now we investigate the productivity direct hours ZVW / uren_netto. 

In [0]:
# Use df_direct_hours and df_daily_prod and left join on date
df_combined = pd.merge(
    df_direct_hours, 
    df_daily_prod, 
    on='rapportagedatum', 
    how='left',
    suffixes=('_direct', '_netto')
)

# calculate ZVW productivity
df_combined['productiviteit_percentage_7d'] = (
    df_combined['uren_7d_gemiddelde_direct']
     / df_combined['uren_7d_gemiddelde_netto']
) * 100

# Plot productivity
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 6))

sns.lineplot(
    data=df_combined, 
    x='rapportagedatum', 
    y='productiviteit_percentage_7d', 
    color='#d62728', 
    linewidth=2.5,
    label='Productivity percentage (7-Day Trend)'
)

plt.title('Productiviteit over time', fontsize=14, pad=15)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Productivity (%)', fontsize=12)
plt.legend()
plt.show()

Based on this plot, productivity is very low. However, only certain functions can produce revenue. These functions can be filtered using the Functiegroep "Behandeling".



In [0]:
# Group uren_netto by date
daily_productivity_behandeling = (productiviteit
                                  .filter(F.col("functie_groep") == "Behandeling")
                                  .groupBy("rapportagedatum")
                                  .agg(F.round(F.sum("uren_netto"), 2).alias("totale_uren_netto"))
                                  .orderBy("rapportagedatum"))

# Convert to pandas
df_daily_prod_beh = daily_productivity_behandeling.toPandas()

# Convert to date
df_daily_prod_beh['rapportagedatum'] = pd.to_datetime(df_daily_prod_beh['rapportagedatum'])

# Calculate MA 7-day
df_daily_prod_beh['uren_7d_gemiddelde'] = df_daily_prod_beh['totale_uren_netto'].rolling(window=7).mean()

# Use df_direct_hours and df_daily_prod_beh and left join on date
df_combined = pd.merge(
    df_direct_hours, 
    df_daily_prod_beh, 
    on='rapportagedatum', 
    how='left',
    suffixes=('_direct', '_netto')
)

# calculate ZVW productivity
df_combined['productiviteit_percentage_7d'] = (
    df_combined['uren_7d_gemiddelde_direct']
     / df_combined['uren_7d_gemiddelde_netto']
) * 100

# Plot productivity
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 6))

sns.lineplot(
    data=df_combined, 
    x='rapportagedatum', 
    y='productiviteit_percentage_7d', 
    color='#d62728', 
    linewidth=2.5,
    label='Productivity percentage (7-Day Trend)'
)

plt.title('Productiviteit over time', fontsize=14, pad=15)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Productivity (%)', fontsize=12)
plt.legend()
plt.show()

Similar trend outcome, productivity in direct hours is lower in later years. Could be an increase in indirect work, non-declarable or non-productive hours, which is interesting to investigate for finding the answer why the productivity is dropping. However, we are searching for variables which influence the revenue directly and we should therefore focus on the underlying mechanism for generating revenue. 

Generating revenue in mental health care in the Netherlands is done by using 'declaratiecodes'. Each ZVW declaratiecode can be found in the dataset "Tarieven". Each declaratiecode is a result of a specific combination of "Declaratiecode_categorie", "Beroepscategorie", "Setting", "Consulttype", "tijdrange", "groepsgrootte" and/or "zorgzwaarte". Therefore, it is much more interesting to know when each declaratiecode and which categories produce the most revenue.

In [0]:
def plot_revenue_bar(column_name, title_name, target_year=2025):
    
    # Hulpfunction
    def safe_select(df, col):
        year_filtered_df = df.filter(F.year("rapportagedatum") == target_year)
        if col in df.columns:
            return year_filtered_df.select(col, "tarief_prijspeil_2026")
        else:
            return year_filtered_df.select(F.lit(None).cast("string").alias(col), "tarief_prijspeil_2026")

    # Filter and combine datasets
    act_subset = safe_select(activiteiten_zvw, column_name)
    ozp_subset = safe_select(ozp_zvw, column_name)
    verblijf_subset = safe_select(verblijf_zvw, column_name)
    
    combined_zvw = act_subset.union(ozp_subset).union(verblijf_subset)
    
    # Filter null values
    combined_zvw = combined_zvw.filter(F.col(column_name).isNotNull())
    
    # Groupby
    revenue_per_category_spark = (
        combined_zvw
        .groupBy(column_name)
        .agg(F.round(F.sum("tarief_prijspeil_2026"), 2).alias("Total_Revenue"))
        .orderBy(F.col("Total_Revenue").desc())
    )
    
    # Convert to Pandas
    df_category = revenue_per_category_spark.toPandas()
    
    # Safety check if there is no data
    if df_category.empty:
        print(f"No data found for {title_name} in {target_year}.")
        return

    # Plot
    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(14, 8))
    
    sns.barplot(
        data=df_category,
        x='Total_Revenue',
        y=column_name,
        palette='Blues_r'
    )
    
    # formatting to M
    def miljoenen_formatter(x, pos):
        return f"€ {x*1e-6:.1f}M"
    plt.gca().xaxis.set_major_formatter(FuncFormatter(miljoenen_formatter))
    
    plt.title(f'Total Revenue by {title_name} ({target_year}, Prijspeil 2026)', fontsize=16, pad=20)
    plt.xlabel('Total Revenue (in M)', fontsize=12)
    plt.ylabel(title_name, fontsize=12)
    
    plt.tight_layout() 
    plt.show()

In [0]:
plot_revenue_bar("declaratiecode_categorie", "Declaratiecode Category")

It's clear that Consult and Verblijf generate the most Revenue. These will be very important to predict.

In [0]:
plot_revenue_bar("beroepscategorie", "Professional Category")

In [0]:
plot_revenue_bar("setting", "Setting")

In [0]:
def plot_over_time(column_name, title_name):
    
    # Hulpfunctie: Check of de kolom bestaat, zo niet, vul met Null
    def safe_select(df, col):
        if col in df.columns:
            return df.select("rapportagedatum", col, "tarief_prijspeil_2026")
        else:
            return df.select("rapportagedatum", F.lit(None).cast("string").alias(col), "tarief_prijspeil_2026")

    # Select data and combine
    act_subset = safe_select(activiteiten_zvw, column_name)
    ozp_subset = safe_select(ozp_zvw, column_name)
    verblijf_subset = safe_select(verblijf_zvw, column_name)
    
    combined_zvw = act_subset.union(ozp_subset).union(verblijf_subset)
    
    # Filter Null waarden
    combined_zvw = combined_zvw.filter(F.col(column_name).isNotNull())
    
    # Groupby
    monthly_spark = (
        combined_zvw
        .withColumn("maand", F.trunc("rapportagedatum", "month"))
        .groupBy("maand", column_name)
        .agg(F.round(F.sum("tarief_prijspeil_2026"), 2).alias("totale_omzet"))
    )
    
    # Convert to pandas
    df_monthly = monthly_spark.toPandas()
    df_monthly['maand'] = pd.to_datetime(df_monthly['maand'])
    
    # Plot
    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(15, 6))
    
    sns.lineplot(
        data=df_monthly, 
        x='maand', 
        y='totale_omzet', 
        hue=column_name, 
        marker='o', 
        linewidth=2.5,
        palette='tab10'
    )
    
    def miljoenen_formatter(x, pos):
        return f"€ {x*1e-6:.1f}M"
    plt.gca().yaxis.set_major_formatter(FuncFormatter(miljoenen_formatter))
    
    plt.title(f'Monthly Revenue Trend: Top 5 {title_name} (Prijspeil 2026)', fontsize=16, pad=20)
    plt.xlabel('Date (Monthly)', fontsize=12)
    plt.ylabel('Total Revenue', fontsize=12)
    
    plt.legend(title=title_name, bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()

In [0]:
plot_over_time("declaratiecode_categorie", "Declaratiecode Category")

No big changes, apart from thee fact that Verblijf increased

In [0]:
plot_over_time("beroepscategorie", "Professional Category")

Overige beroepen en Arts - Specialist slightly increased

In [0]:
plot_over_time("setting", "Setting")

Outreachend is slightly decreasing

In [0]:
plot_over_time("org_level_1", "Cluster")

we can also see a clear increase in revenue at Klinieken, which correlates with the increase in Verblijf. Verblijf is only realised at the Klinieken.

In [0]:
plot_over_time("consult_type", "Consult Type")

We can also see a clear increase in Diagnostiek and a clear decrease in Behandeling. This is a result of a lecture 'tijdschrijven', where the company gave indications when to write Diagnostiek and when to write Behandeling. This is also a reason why productivity has decreased: Diagnostiek takes a lot more administration time. 

In [0]:
from IPython.display import display, HTML

kolom_mapping = {
    "jaar": "jaar",
    "maand": "maand",
    "soort_ggz": "soort_ggz_zorg",
    "declarabel": "declarabel",
    "declaratiecode_categorie": "declaratiecode_categorie",
    "declaratiecode": "declaratiecode",
    "activiteitcode": "activiteitcode",
    "crisis_binnen_budget": "crisis_binnen_budget",
    "financier": "financier",
    "betalende_instantie": "betalende_instantie", 
    "bedrag_totaal": "bedrag_totaal"
}

def selecteer_veilig(df, mapping):
    kolommen_voor_select = []
    for doel_naam, bron_naam in mapping.items():
        if bron_naam in df.columns:
            kolommen_voor_select.append(F.col(bron_naam).alias(doel_naam))
        else:
            # Kolom ontbreekt in deze tabel? Voeg hem toe als leeg (null)
            kolommen_voor_select.append(F.lit(None).alias(doel_naam))
    return df.select(*kolommen_voor_select)

# FILTER TOEGEVOEGD: We filteren de dataframes direct op jaar 2024 en 2025 voordat we selecteren
df_act_veilig  = selecteer_veilig(activiteiten.filter(F.col("jaar").isin(2024, 2025)), kolom_mapping)
df_ozp_veilig  = selecteer_veilig(ozp.filter(F.col("jaar").isin(2024, 2025)), kolom_mapping)
df_verb_veilig = selecteer_veilig(verblijf.filter(F.col("jaar").isin(2024, 2025)), kolom_mapping)

df_gecombineerd = df_act_veilig.unionByName(df_ozp_veilig).unionByName(df_verb_veilig)

groep_kolommen = [k for k in kolom_mapping.keys() if k != "bedrag_totaal"]

df_resultaat = (
    df_gecombineerd
    .groupBy(*groep_kolommen)
    .agg(F.sum("bedrag_totaal").alias("totaal_bedrag"))
)

from datetime import datetime

# 7. Exporteren naar je lokale Workspace map
df_pandas = df_resultaat.toPandas()

# Genereer een unieke datum/tijd stempel (bijv. 20260413_1730)
nu = datetime.now().strftime("%Y%m%d_%H%M")
bestand_naam = f"omzet_export_{nu}.xlsx"

# Opslaan met de unieke naam
df_pandas.to_excel(bestand_naam, index=False, engine='openpyxl')

print(f"✅ Bestand is succesvol opgeslagen als: {bestand_naam}")
print("Ververs eventueel de map aan de linkerkant om het nieuwe bestand te zien verschijnen!")

In [0]:
%pip install openpyxl